# Spaceship Titanic

Predict which passengers were transported to an alternate dimension.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

## 1. Load Data

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

train.shape, test.shape

In [ ]:
train.head()

In [ ]:
train.info()

## 2. EDA

In [ ]:
numerical_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
sns.pairplot(train, vars=numerical_cols, hue="Transported", diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10}, corner=True, height=2.5)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train[numerical_cols + ["Transported"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

In [ ]:
categorical_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]

fig, axes = plt.subplots(1, len(categorical_cols), figsize=(16, 4))
for ax, col in zip(axes, categorical_cols):
    train.groupby(col)["Transported"].mean().plot.bar(ax=ax, rot=0)
    ax.set_ylabel("Transport rate")
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.axhline(train["Transported"].mean(), color="red", linestyle="--", label="overall")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_cols:
    print(f"--- {col} ---")
    print(train[col].value_counts(dropna=False))
    print()

In [ ]:
train["Cabin"].head(10)

In [ ]:
train["Transported"].value_counts().plot.bar(rot=0)
plt.title("Target balance")
plt.show()

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"{'Column':<15} {'Missing':>7} {'%':>6}")
print("-" * 30)
for col, count in missing.items():
    print(f"{col:<15} {count:>7} {count / len(train) * 100:>5.1f}%")

In [ ]:
train.isnull().sum(axis=1).value_counts().sort_index()

## 3. Preprocessing & Feature Engineering

In [ ]:
bool_cols = ["Transported", "CryoSleep", "VIP"]
for col in bool_cols:
    train[col] = train[col].map({True: 1, False: 0, "True": 1, "False": 0})

train[bool_cols].head()

In [ ]:
train["PassengerId"] = train["PassengerId"].str.split("_").str[0].astype(int)

train["PassengerId"].head()

In [ ]:
train = pd.get_dummies(train, columns=["HomePlanet", "Destination"], dtype=int)

train.head()

In [ ]:
cabin_split = train["Cabin"].str.split("/", expand=True)
train["CabinDeck"] = cabin_split[0]
train["CabinNum"] = cabin_split[1].astype(float)
train["CabinSide"] = cabin_split[2].map({"P": 0, "S": 1})
train.drop(columns=["Cabin"], inplace=True)

train = pd.get_dummies(train, columns=["CabinDeck"], dtype=int)

train[["CabinNum", "CabinSide"] + [c for c in train.columns if c.startswith("CabinDeck_")]].head()

In [ ]:
train["LastName"] = train["Name"].str.split().str[-1]
train.drop(columns=["Name"], inplace=True)

train["LastName"].head()

In [ ]:
train["FamilySize"] = train.groupby("LastName")["LastName"].transform("count")
train.drop(columns=["LastName"], inplace=True)

train["FamilySize"].value_counts().sort_index()

In [ ]:
median_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck", "CabinNum", "FamilySize"]
mode_cols = ["CryoSleep", "VIP", "CabinSide"]

for col in median_cols:
    train[col] = train[col].fillna(train[col].median())
for col in mode_cols:
    train[col] = train[col].fillna(train[col].mode()[0])

print("Remaining NaNs:", train.isnull().sum().sum())

## 4. Baseline Model

## 5. Improved Model

## 6. Submission